# 🛤️ Day 05: Tree Path Problems — The Interview Favorites

---

## Path Problems: Two Categories

1. **Root-to-leaf paths:** Start at root, end at a leaf. Pass info top-down.
2. **Any-to-any paths:** Can start/end anywhere. Need bottom-up info.

The second category (diameter, max path sum) is trickier because at each node, the path might go through that node connecting left and right subtrees.

---

In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

from collections import deque

def build_tree(values):
    if not values:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

## 🎯 Problem 1: Path Sum (LC #112) & Path Sum II (LC #113)

**Path Sum:** Does any root-to-leaf path sum to a target?

**The approach:** Subtract current value from target as you go. At a leaf, check if remaining target is 0.

**Path Sum II:** Find ALL root-to-leaf paths that sum to target. Same idea but collect paths.

In [ ]:
def hasPathSum(root, targetSum):
    """Does any root-to-leaf path sum to target?"""
    if not root:
        return False
    # Leaf check
    if not root.left and not root.right:
        return root.val == targetSum
    # Recurse with reduced target
    remaining = targetSum - root.val
    return hasPathSum(root.left, remaining) or hasPathSum(root.right, remaining)

def pathSum(root, targetSum):
    """Return ALL paths that sum to target."""
    result = []
    
    def dfs(node, remaining, path):
        if not node:
            return
        path.append(node.val)
        
        if not node.left and not node.right and remaining == node.val:
            result.append(path[:])  # Copy the path!
        
        dfs(node.left, remaining - node.val, path)
        dfs(node.right, remaining - node.val, path)
        path.pop()  # Backtrack
    
    dfs(root, targetSum, [])
    return result

tree = build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1])
print(f"Has path sum 22: {hasPathSum(tree, 22)}")  # True
print(f"All paths summing to 22: {pathSum(tree, 22)}")  # [[5,4,11,2], [5,8,4,5]]

**Key detail in Path Sum II:** `result.append(path[:])` — you must copy the path! Otherwise all entries point to the same (eventually empty) list.

---

## 🎯 Problem 2: Diameter of Binary Tree (LC #543)

The diameter is the longest path between **any** two nodes (measured in edges). This path may or may not pass through the root.

**Key insight:** At every node, the longest path through it is `left_height + right_height`. We track the global maximum while computing heights bottom-up.

In [ ]:
def diameterOfBinaryTree(root):
    """
    At each node: diameter through it = left_height + right_height.
    Return max across all nodes.
    Time: O(n), Space: O(h)
    """
    diameter = [0]  # Using list for mutation in nested function
    
    def height(node):
        if not node:
            return 0
        left_h = height(node.left)
        right_h = height(node.right)
        
        # Update diameter: path going through this node
        diameter[0] = max(diameter[0], left_h + right_h)
        
        return 1 + max(left_h, right_h)
    
    height(root)
    return diameter[0]

tree = build_tree([1, 2, 3, 4, 5])
print(f"Diameter: {diameterOfBinaryTree(tree)}")  # 3 (path: 4→2→1→3 or 5→2→1→3)

**This pattern is crucial.** The function returns one thing (height) but updates a global variable with something else (diameter). You'll see this same pattern in Maximum Path Sum.

---

## 🎯 Problem 3: Binary Tree Maximum Path Sum (LC #124)

Find the maximum sum of **any** path (node to node). This is a hard problem but uses the same pattern as diameter.

**Difference from diameter:** Values can be negative, so sometimes it's better NOT to include a subtree.

At each node:
- Max gain from left subtree (0 if negative — don't include it)
- Max gain from right subtree (0 if negative)
- Path through this node = left_gain + node.val + right_gain
- But when returning to parent, can only choose ONE side (path can't fork)

In [ ]:
def maxPathSum(root):
    """
    Same pattern as diameter but with values.
    Time: O(n), Space: O(h)
    """
    max_sum = [float('-inf')]
    
    def gain(node):
        if not node:
            return 0
        
        # Max gain from each subtree (0 if negative = don't take it)
        left_gain = max(gain(node.left), 0)
        right_gain = max(gain(node.right), 0)
        
        # Path through this node (using both sides)
        path_sum = left_gain + node.val + right_gain
        max_sum[0] = max(max_sum[0], path_sum)
        
        # Return to parent: can only go one direction
        return node.val + max(left_gain, right_gain)
    
    gain(root)
    return max_sum[0]

tree = build_tree([-10, 9, 20, None, None, 15, 7])
print(f"Max path sum: {maxPathSum(tree)}")  # 42 (15 → 20 → 7)

---

## 🎯 Problem 4: Lowest Common Ancestor (LC #236)

Find the LCA of two nodes p and q. The LCA is the deepest node that is an ancestor of both.

**Beautiful recursive logic:**
- If current node is None, or is p, or is q → return it
- Recurse left and right
- If both sides return non-None → current node is the LCA
- If only one side returns non-None → that side has the answer

In [ ]:
def lowestCommonAncestor(root, p, q):
    """
    If p and q are on different sides, current node is LCA.
    Time: O(n), Space: O(h)
    """
    if not root or root == p or root == q:
        return root
    
    left = lowestCommonAncestor(root.left, p, q)
    right = lowestCommonAncestor(root.right, p, q)
    
    if left and right:
        return root       # p and q on different sides → LCA
    return left or right  # Both on one side

# Build tree and find LCA
tree = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
# Find nodes 5 and 1
p = tree.left       # Node 5
q = tree.right      # Node 1
lca = lowestCommonAncestor(tree, p, q)
print(f"LCA of 5 and 1: {lca.val}")  # 3

# Find nodes 5 and 4
q2 = tree.left.right.right  # Node 4
lca2 = lowestCommonAncestor(tree, p, q2)
print(f"LCA of 5 and 4: {lca2.val}")  # 5 (5 is ancestor of 4)

**LCA is one of the most elegant tree algorithms.** Just 5 lines of code, but every line matters.

---

## 🗺️ Path Problem Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  TREE PATH PATTERNS                                                 ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. ROOT-TO-LEAF: Pass remaining sum down, check at leaf            ║
║     Collect paths: backtracking with path.append() / path.pop()     ║
║                                                                      ║
║  2. ANY-TO-ANY (Diameter, Max Path Sum):                            ║
║     Function returns: max single-direction contribution             ║
║     Global tracks:    max path through each node (both sides)       ║
║     Key: return one side, but CHECK both sides                      ║
║                                                                      ║
║  3. LCA: If both sides found → current is LCA                      ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Key Idea |
|---|---------|------------|----------|
| 112 | Path Sum | Easy | Subtract going down, check leaf |
| 113 | Path Sum II | Medium | Backtracking, copy path |
| 543 | Diameter of Binary Tree | Easy | Height + global max |
| 124 | Binary Tree Max Path Sum | Hard | Gain with max(0), same pattern |
| 236 | Lowest Common Ancestor | Medium | Split = found LCA |
| 257 | Binary Tree Paths | Easy | Root-to-leaf, string building |
| 437 | Path Sum III | Medium | Prefix sum on tree paths |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Root-to-leaf: pass remaining target down, check at leaf        │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Diameter/MaxPath: return single-side, track global both-sides  │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Negative gains: use max(gain, 0) to optionally skip subtrees  │
├──────────────────────────────────────────────────────────────────────┤
│  📌 LCA: 5 lines. If both sides non-null → current node is LCA   │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Tree Views

Right side view, boundary traversal, and vertical order.